# Movie recommendations on MovieLens 1M
Four recommenders, evaluated the way production teams do it: hide each user's most recent movie and check
whether the model ranks it in the top 10. Run everything with **Runtime → Run all** (about 2 minutes, CPU is fine).

## 1. Get the code and the data

In [ ]:
!git clone -q https://github.com/siri780/movielens-recommender.git repo 2>/dev/null || (cd repo && git pull -q)
!curl -sL -o ml-1m.zip https://files.grouplens.org/datasets/movielens/ml-1m.zip && unzip -oq ml-1m.zip
import sys; sys.path.insert(0, "repo/src")
import recsys
ratings, movies = recsys.load_movielens("ml-1m")
print(f"{len(ratings):,} ratings from {ratings.user.nunique():,} users on {ratings.item.nunique():,} movies")

## 2. Leave-last-out split
Each user's most recent movie becomes the test item; everything before it is training data.

In [ ]:
train, test = recsys.split_last_out(ratings)
X, test_u, test_i, items = recsys.to_matrix(train, test)
print(f"train matrix: {X.shape[0]:,} users x {X.shape[1]:,} movies, {X.nnz:,} interactions ({X.nnz/(X.shape[0]*X.shape[1]):.1%} filled)")

## 3. Train and evaluate all four models

In [ ]:
table, per_user = recsys.run(X, test_u, test_i)
table.round(4)

## 4. Is the best model really better? (bootstrap over users)

In [ ]:
ranked = table["NDCG@10"].sort_values(ascending=False).index
for a, b in [(ranked[0], ranked[1]), (ranked[0], "Popularity")]:
    m, lo, hi = recsys.bootstrap_diff(per_user[a], per_user[b])
    print(f"{a} vs {b}: NDCG@10 {m:+.4f}  (95% CI {lo:+.4f} to {hi:+.4f})")

## 5. Example: what EASE recommends for one user

In [ ]:
import numpy as np
title = movies.set_index("item")["title"]
u = test_u[0]
watched = train[train.user == train.user.unique()[u]].sort_values("ts").tail(5)["item"]
scores = recsys.ease(X, np.array([u]))[0]
scores[X[u].toarray()[0] > 0] = -np.inf
top = items[np.argsort(-scores)[:10]]
print("Recently watched:"); print("  " + "\n  ".join(title[watched].tolist()))
print("\nRecommended:");   print("  " + "\n  ".join(title[top].tolist()))
print("\nHidden next movie:", title[items[test_i[0]]])